# 3.0 모델링
2.0에서 만든 연간 정답 표로 다음 해 잘버팀을 예측한다.
- 모델 고르기: 결과연도 2023 학습 → 2024 검증
- 최종 시험: 2023~2024 학습 → 2025 (딱 한 번)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cafe_target_annual_2022_2025.csv")
print(df.shape)

(3504, 30)


## 1. 입력 칸 정하기와 연도별 나누기 ✅
입력: 작년 변화율, 작년에 카페가 유동인구보다 덜 늘었나, 폐업률, 상권 유형 (2.0 EDA 결론의 후보)
- 빈칸이 있는 행을 빼서 결과연도별 2023: 863행, 2024: 874행, 2025: 865행 (2022는 작년 값이 없어 전부 빠짐)
- 학습 2023 / 검증 2024 / 최종 시험 2025 (시험 연도는 마지막에 한 번만 봄)
- 정답 칸(다음 해 매출, 변화율, 중앙값, 잘버팀)과 상권 코드·이름은 입력에서 뺌

In [2]:
# H1 예측 버전 칸 다시 만들기 (2.0에서는 탐색용 표에만 만들었음)
df["작년_카페가_덜_늘었나"] = (df["작년_카페_증가율"] < df["작년_유동인구_증가율"]).astype(int)
df.loc[df["작년_카페_증가율"].isna() | df["작년_유동인구_증가율"].isna(), "작년_카페가_덜_늘었나"] = np.nan

입력칸 = ["작년_변화율", "작년_카페가_덜_늘었나", "폐업률", "상권_구분_코드_명"]   # ★ 모델이 보는 칸
df["상권_구분_코드_명"] = df["상권_구분_코드_명"].replace({"관광특구": "발달상권"})   # 사례가 적은 관광특구는 비슷한 발달상권과 합침
쓸행 = df.dropna(subset=입력칸).copy()
print(쓸행.groupby("결과연도").size())

train = 쓸행[쓸행["결과연도"] == 2023]     # 학습
val = 쓸행[쓸행["결과연도"] == 2024]       # 검증 (모델 고르기)
test = 쓸행[쓸행["결과연도"] == 2025]      # ★ 최종 시험: 마지막까지 열어 보지 않음
print(train.shape, val.shape)
print("검증 연도 잘버팀 비율:", round(val["잘버팀"].mean(), 3))

결과연도
2023    863
2024    874
2025    865
dtype: int64
(863, 31) (874, 31)
검증 연도 잘버팀 비율: 0.503


## 2. 기준선 ✅
- 동전 던지기: 약 50% (검증 연도 실제 잘버팀 비율 50.3%)
- 작년 변화율 규칙(가운데보다 높으면 1): 정확도 55.0%
- 작년 변화율 상위 20%를 뽑으면 62.3%가 실제로 잘 버팀 → 모델은 이 62.3%를 넘어야 의미가 있음
- 주 점수는 상위 20% 적중률: 추천받은 상권이 실제로 괜찮은지가 목적에 더 맞음

In [3]:
# 상위 20% 적중률: 점수가 높은 20% 상권 중 실제로 잘 버틴 비율
def 상위20(점수, 정답):
    기준 = 점수.quantile(0.8)              # 위에서 20% 지점 값
    return 정답[점수 >= 기준].mean()

# 기준선: 작년 변화율이 가운데보다 높으면 '잘버팀'이라고 찍기
규칙 = (val["작년_변화율"] > val["작년_변화율"].median()).astype(int)
print("작년 변화율 규칙 - 정확도:", round((규칙 == val["잘버팀"]).mean(), 3))
print("작년 변화율 규칙 - 상위 20% 적중률:", round(상위20(val["작년_변화율"], val["잘버팀"]), 3))

작년 변화율 규칙 - 정확도: 0.55
작년 변화율 규칙 - 상위 20% 적중률: 0.623


## 3. 로지스틱 회귀 ✅
입력 4개로 2023년을 학습하고 2024년에 검증. 관광특구는 사례가 적어 발달상권과 합침.
- 4개 칸 모델: 정확도 54.3%, 상위 20% 적중률 60.0% → 기준선(작년 변화율 규칙 62.3%)보다 낮음
- 가중치: 작년 변화율 0.27, 폐업률 0.21, 카페가 덜 늘었나 0.16 (모두 +), 발달상권 +, 골목·전통시장 −
- 칸을 하나씩 더하면 상위 20%: 62.3% → 58.3% → 58.9% → 60.0%
- 상위 20%는 약 175곳이라 ±4%p는 우연으로도 생김 → 칸을 더해서 나아졌다고 볼 수 없음
- 관광특구를 합치기 전(59.4%)과 차이 없음

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

숫자칸 = ["작년_변화율", "작년_카페가_덜_늘었나", "폐업률"]
글자칸 = ["상권_구분_코드_명"]

# 모델 만드는 틀: 숫자는 크기 맞추기, 글자는 0/1 칸으로 바꾸기 → 로지스틱 회귀
def 만들기(숫자, 글자):
    준비 = ColumnTransformer(
        [("숫자", StandardScaler(), 숫자)]
        + ([("글자", OneHotEncoder(handle_unknown="ignore"), 글자)] if 글자 else []))
    return make_pipeline(준비, LogisticRegression(max_iter=1000))

모델 = 만들기(숫자칸, 글자칸)
모델.fit(train[입력칸], train["잘버팀"])                                    # ★ 2023년으로 학습
확률 = pd.Series(모델.predict_proba(val[입력칸])[:, 1], index=val.index)     # 2024년 상권마다 잘버팀 확률
print("로지스틱 - 정확도:", round((모델.predict(val[입력칸]) == val["잘버팀"]).mean(), 3))
print("로지스틱 - 상위 20% 적중률:", round(상위20(확률, val["잘버팀"]), 3))   # ★ 62.3%와 비교

# 칸마다 배운 가중치 (+면 잘버팀 쪽, −면 반대 쪽)
print(pd.Series(모델[-1].coef_[0], index=모델[0].get_feature_names_out()).round(3))

로지스틱 - 정확도: 0.543
로지스틱 - 상위 20% 적중률: 0.6
숫자__작년_변화율             0.267
숫자__작년_카페가_덜_늘었나       0.162
숫자__폐업률                0.212
글자__상권_구분_코드_명_골목상권   -0.084
글자__상권_구분_코드_명_발달상권    0.268
글자__상권_구분_코드_명_전통시장   -0.189
dtype: float64


In [5]:
# 본인이 정한 순서대로 칸을 하나씩 더하며 비교
조합 = [
    (["작년_변화율"], []),
    (["작년_변화율", "작년_카페가_덜_늘었나"], []),
    (["작년_변화율", "작년_카페가_덜_늘었나", "폐업률"], []),
    (숫자칸, 글자칸),
]
for 숫자, 글자 in 조합:
    m = 만들기(숫자, 글자)
    m.fit(train[숫자 + 글자], train["잘버팀"])
    p = pd.Series(m.predict_proba(val[숫자 + 글자])[:, 1], index=val.index)
    print(숫자 + 글자, "정확도:", round((m.predict(val[숫자 + 글자]) == val["잘버팀"]).mean(), 3),
          "/ 상위 20%:", round(상위20(p, val["잘버팀"]), 3))

['작년_변화율'] 정확도: 0.535 / 상위 20%: 0.623
['작년_변화율', '작년_카페가_덜_늘었나'] 정확도: 0.541 / 상위 20%: 0.583
['작년_변화율', '작년_카페가_덜_늘었나', '폐업률'] 정확도: 0.566 / 상위 20%: 0.589
['작년_변화율', '작년_카페가_덜_늘었나', '폐업률', '상권_구분_코드_명'] 정확도: 0.543 / 상위 20%: 0.6


## 4. 랜덤포레스트 ✅
같은 입력 4개로 2023 학습 → 2024 검증. 관광특구는 발달상권과 합침.
- 최소행 20: 상위 20% 65.1%, 개업률을 더하면 64.0% → 개업률은 도움이 안 됨
- 최소행만 바꿔도 57.7%(5) ~ 65.1%(20) ~ 62.3%(50)로 크게 흔들림
- 20은 2024년 결과를 본 뒤의 값이라 믿을 수 없음 → 설정은 2023년 교차검증으로 다시 고름
- 교차검증(2023년 5묶음): 최소행 50이 64.0%로 가장 높고 흔들림도 작음 → 최종 설정 50
- 최소행 50의 2024년 검증: 62.3% = 기준선과 같음
- 결론: 로지스틱, 랜덤포레스트 모두 작년 변화율 규칙을 확실히 넘지 못함

In [6]:
from sklearn.ensemble import RandomForestClassifier

# 숲 만드는 틀: 숫자는 그대로, 글자는 0/1 칸으로 → 랜덤포레스트
def 숲(숫자, 글자, 최소행=20):
    준비 = ColumnTransformer(
        [("숫자", "passthrough", 숫자)]
        + ([("글자", OneHotEncoder(handle_unknown="ignore"), 글자)] if 글자 else []))
    return make_pipeline(준비, RandomForestClassifier(n_estimators=500, min_samples_leaf=최소행, random_state=42))

for 숫자 in [숫자칸, 숫자칸 + ["개업률"]]:          # 입력 4개, 그리고 개업률을 더한 5개
    m = 숲(숫자, 글자칸)
    m.fit(train[숫자 + 글자칸], train["잘버팀"])
    p = pd.Series(m.predict_proba(val[숫자 + 글자칸])[:, 1], index=val.index)
    print(숫자 + 글자칸, "정확도:", round((m.predict(val[숫자 + 글자칸]) == val["잘버팀"]).mean(), 3),
          "/ 상위 20%:", round(상위20(p, val["잘버팀"]), 3))

# 마지막 숲(5개 칸)에서 칸마다 얼마나 많이 쓰였나
print(pd.Series(m[-1].feature_importances_, index=m[0].get_feature_names_out()).round(3))

['작년_변화율', '작년_카페가_덜_늘었나', '폐업률', '상권_구분_코드_명'] 정확도: 0.559 / 상위 20%: 0.651
['작년_변화율', '작년_카페가_덜_늘었나', '폐업률', '개업률', '상권_구분_코드_명'] 정확도: 0.553 / 상위 20%: 0.64
숫자__작년_변화율             0.363
숫자__작년_카페가_덜_늘었나       0.059
숫자__폐업률                0.277
숫자__개업률                0.228
글자__상권_구분_코드_명_골목상권    0.025
글자__상권_구분_코드_명_발달상권    0.040
글자__상권_구분_코드_명_전통시장    0.007
dtype: float64


In [7]:
# 최소행 5 / 20 / 50으로 바꿔 보기 (고르기용이 아니라 흔들림 확인용)
for 최소행 in [5, 20, 50]:
    m = 숲(숫자칸, 글자칸, 최소행)
    m.fit(train[입력칸], train["잘버팀"])
    p = pd.Series(m.predict_proba(val[입력칸])[:, 1], index=val.index)
    print("최소행", 최소행, "정확도:", round((m.predict(val[입력칸]) == val["잘버팀"]).mean(), 3),
          "/ 상위 20%:", round(상위20(p, val["잘버팀"]), 3))

최소행 5 정확도: 0.543 / 상위 20%: 0.577
최소행 20 정확도: 0.559 / 상위 20%: 0.651
최소행 50 정확도: 0.556 / 상위 20%: 0.623


In [8]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# 상위 20% 적중률을 교차검증 점수로 쓰기 위한 함수
def 상위20점수(모델, X, y):
    p = pd.Series(모델.predict_proba(X)[:, 1], index=X.index)
    return 상위20(p, y)

묶음 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)    # 2023년을 5묶음으로
for 최소행 in [5, 10, 20, 50, 100]:
    점수 = cross_val_score(숲(숫자칸, 글자칸, 최소행), train[입력칸], train["잘버팀"], cv=묶음, scoring=상위20점수)
    print("최소행", 최소행, "교차검증 상위 20% 평균:", round(점수.mean(), 3), "/ 흔들림:", round(점수.std(), 3))   # ★ 2023년 안에서만 비교

최소행 5 교차검증 상위 20% 평균: 0.606 / 흔들림: 0.046
최소행 10 교차검증 상위 20% 평균: 0.611 / 흔들림: 0.014
최소행 20 교차검증 상위 20% 평균: 0.629 / 흔들림: 0.036
최소행 50 교차검증 상위 20% 평균: 0.64 / 흔들림: 0.034
최소행 100 교차검증 상위 20% 평균: 0.631 / 흔들림: 0.033
